# XNAT-Jupyter-CS demo workflow

Important: Launch Jupyter notebook from XNAT project level to run this notebook.

This notebook demonstrates how to:
* analyze an XNAT project for structural scans and segmentations, 
* build a list of scans to process with custom user analyses,
* develop a custom user meta-script to run for each of the scans/sessions,
* launch containerized batch processing for all scans.

## 1. Mandatory user-settable variables
Edit this cell to set all mandatory notebook variables here. 

In [4]:
from pathlib import Path
#set to True to regenerate project directory structure saved in local json file.
rebuild_directory_structure=True

#XNAT project label
project='NSCLCRadiomicsDemo'

#Persistent workspace root path
root_dir=Path("/workspace/mmilchenko")

## 2. General initializations
Just run this cell for notebook environment set up.

In [5]:
import os, subprocess, sys

#Library with XNAT Jupyter workflow Python and shell scripts.
pymipl_path = os.path.abspath('../')
sys.path.append(pymipl_path)
sys.path.append( os.path.abspath(pymipl_path+'/xnat_workflow') )

#dicom_sort is part of pymipl. dicom_sort can automatically analyze XNAT projects for structural scans and segmentations.
from dicom_sort import *

#Derived variable initializations
local_workdir_path=root_dir / project
xnat_project_path=f'/data/projects/{project}/experiments'
directory_structure_file=local_workdir_path / "project_dir_structure.json"
xnat_structure_file=local_workdir_path / "xnat_structure.json"
scanlist_file=local_workdir_path / "scans.csv"


## 3. Create a list of structural scans and associated segmentations.
Run the next cell to Project structure, with all DICOM scans and segmentations, is written to  configuration files in workspace project location.


In [6]:
# analyze_dir finds all structural scans and segmentations (DICOM RTSTRUCT and DICOM Segmentation Object) in the project. 
# The results are saved into a JSON file for quick rerun. 
if rebuild_directory_structure:
    os.makedirs(os.path.dirname(directory_structure_file), exist_ok=True)    
    d=analyze_dir(xnat_project_path,directory_structure_file)
else:
    #load analyzed directory structure from disk
    with open(directory_structure_file, 'r') as file:
        d = json.load(file)

#This writes out human-readable list of structural scans with segmentations into a csv file.
subjects,scans=reindex_to_structurals_and_segs(d,xnat_structure_file,scanlist_file)
print (f'Number of structural scans: {len(scans)}')
print ('First scan: ',scans[0])

/data/projects/NSCLCRadiomicsDemo/experiments/09-18-2008-StudyID-NA-69331
/data/projects/NSCLCRadiomicsDemo/experiments/01-01-2014-StudyID-NA-85095
/data/projects/NSCLCRadiomicsDemo/experiments/01-01-2014-StudyID-NA-34270
Number of structural scans: 3
First scan:  {'Subject': 'LUNG1-001', 'Experiment': '09-18-2008-StudyID-NA-69331', 'StructScan': '0', 'StructScanSerDesc': '', 'SegScan1': '3', 'SegScan1_SerDesc': '', 'SegScan1_SOPClass': 'RTStruct', 'SegScan2': '300', 'SegScan2_SerDesc': 'Segmentation', 'SegScan2_SOPClass': 'Seg'}


## 4. Workflow inititalizations

In [18]:
import workflow_adapters as wa
import importlib
importlib.reload(wa)

#env_type controls mode for the environment where we build the scripts.
#jupyter: the mode where all processing is run in a single dedicated XNAT-Jupyter environment
#container: the mode all processing will be run in dedicated containers, one container per scan or session

env_type='jupyter'
#env_type='container'

#This is the label of a session resource where generated job scripts, logs and output will be saved.
workflow_id='lungmask'

# Command ID of the dedicated container command. Shown on top of json command edit screen 'edit command'. 
xnat_command_id=20

# Command name of the dedicated container command. Taken from the first 'name' value in the command json.
xnat_command_wrapper_id='xnat-ai-workflow'

#user micromamba environment
user_env_repo='NONE'
#user (re)source directory
user_src_repo='NONE'

#the next section configures global variables that can be used in workflows.

##########################################################################################
# g_project: XNAT project name 

##########################################################################################
# g_workflow_id: taken from workflow_id

##########################################################################################
# g_user_env_repo: the folder name stem of the user-supplied micromamba enironment repository 
# stored as subfolder in the project 'ENVS' resource. Currently not supported due to XNAT limitations.
# if this is set to 'NONE', in container mode an environment built into Docker image (if any) will be used, 
# from the default location /opt/packages/user/user_env. 

##########################################################################################
# g_env_repo_dir: actual location of the the user-supplied micromamba enironment repository. 
# 'jupyter' mode: if g_user_env_repo is set, this is set to 
# /data/projects/<PROJECT>/RESOURCES/ENVS/<g_user_env_repo>
# if g_user_env_repo='NONE', this can be set manually to point to the local environment repo.
# 'container' mode: default to '/opt/packages/user/user_env'. If g_user_env_repo is set, this location 
# is mounted from /data/projects/<PROJECT>/RESOURCES/ENVS/<g_user_env_repo>

##########################################################################################
# g_user_src_repo: folder name stem of the user source code/resources that will be used by the 
# workflow, stored as subfolder in the project 'SRC' resource.
# This can be 'NONE', then no resources directory will be available during runtime.

##########################################################################################
# g_alg_repo_dir: the actual location of the user source code/resources dir. 
# In 'jupyter' mode, default to '/data/projects/<PROJECT>/RESOURCES/SRC/<g_user_src_repo>'.
# Can be overridden if that repo does not exist.
# In 'container' mode, default to '/opt/packages/user/alg_repo'

##########################################################################################
# g_input_mount_path: the input folder with the list of input project experiments, 
# mounted from the XNAT archive.
# for 'jupyter', default to '/data/project/<project>/experiments'
# for 'container', default to '/input

##########################################################################################
# g_local_workdir_path: local workdir where configuration files, logs, scripts, 
# and outputs will be written. 
# in 'jupyter' mode, this should point to a local writable dir.
# in 'container' mode, default to '/workdir'

#########################################################################################
# g_pymipl_dir: local directory with XNAT workflow generator scripts.
# 'jupyter' mode: any local dir where 'pymipl' library is cloned
# 'container' mode: auto-set to /opt/packages/pymipl

#########################################################################################

if env_type=='jupyter':
    global_vars=wa.init_global_vars(env_type, project, workflow_id, g_local_workdir_path=local_workdir_path,
        g_pymipl_dir=root_dir / 'pymipl')
elif env_type=='container':
    global_vars=wa.init_global_vars(env_type, project, workflow_id)

global_vars

{'g_project': 'NSCLCRadiomicsDemo',
 'g_workflow_id': 'lungmask',
 'g_user_env_repo': 'NONE',
 'g_env_repo_dir': 'NONE',
 'g_user_src_repo': 'NONE',
 'g_alg_repo_dir': 'NONE',
 'g_input_mount_path': PosixPath('/data/project/NSCLCRadiomicsDemo/experiments'),
 'g_local_workdir_path': PosixPath('/workspace/mmilchenko/NSCLCRadiomicsDemo'),
 'g_pymipl_dir': PosixPath('/workspace/mmilchenko/pymipl')}

### 4. Create step descriptors for the XNAT workflow parser.
This creates batch files to run on experiment specific containers. Set flags in the beginning to control execution.

Performs the following:
1. convert strcutrual DICOM's to NIFTI
2. convert existing segmentations, if any, to NIFTI
3. run AI segmentation
4. generate QC images
5. compute Dice coefficients.

The batch file is then run outside of this notebook by external containers.

In [19]:
#List keys that can be used inside job scripts as variables.
scans[0].keys()

dict_keys(['Subject', 'Experiment', 'StructScan', 'StructScanSerDesc', 'SegScan1', 'SegScan1_SerDesc', 'SegScan1_SOPClass', 'SegScan2', 'SegScan2_SerDesc', 'SegScan2_SOPClass'])

In [21]:
import importlib
import workflow_adapters as wa
importlib.reload(wa)

wa.set_logger()
dt=datetime.datetime.now().strftime("%Y%m%d_%H%M")
batch_file=local_workdir_path / f"batch_{dt}.sh"
#n indicates starting position in the spreadsheet.
n=0
xnat_interface=None
num_sessions=len(scans)

# start and end positions in the csv file to process.
start_pos=1
end_pos=1

#TODO move these to a separate execution control cell.
nExp=0
nFailed=0
sessions_failed=[]

for scan in scans:
    n=n+1
    if n<start_pos or n>end_pos:  continue

    ###################################################################################################
    # The next block initializes job context.
    job=wa.populate_job_fields(scan)
    
    #Do not change the next two lines to correctly preserve the scan context
    job_scan_context=global_vars['g_input_mount_path'] #/ job_scan_id / 'DICOM'
    if env_type == 'jupyter': job_scan_context = job_scan_context / Path(job_experiment)

    #path where scan DICOM's are mounted.
    job_scan_context = job_scan_context / Path('SCANS')
    #job title to be written to the batch files.
    job['job_title']=f"Workflow {workflow_id}, subject {job['Subject']}, experiment {job['Experiment']}"
    #structural scan DICOM path
    job['job_struct_path'] = job_scan_context / scan['StructScan'] / 'DICOM'
    #assign job id to identify job from logs/outputs
    job_id=f"{workflow_id}_{job['Subject']}_{job['Experiment']}"    
    job['job_id']=job_id
    #job workdir path. 
    job['job_workdir']=global_vars['g_local_workdir_path'] / job['Subject'] / job['Experiment']

    #################################################################################################
    # The next block adds steps to the script that will be executed inside the container. 
    # Edit this block to customize the workflow. 
    step={'title': "1. Run NSCLC tumor segmentation on structural scan"}
    step['command']="micromamba run -p {g_env_repo_dir} python {g_alg_repo_dir}/run_segmentation.py \
        --input {job_struct_path} --output-dir {job_workdir} --output-format nrrd --verbose"
    job['steps']+=[step]

    step={'title': "2. Convert structural scan to NIFTI"}
    step['command']="micromamba run -n base python {g_pymipl_dir}/test_rt-utils.py \
        {job_struct_path} {job_workdir}/ct"
    job['steps']+=[step]
    
    step={'title': "3. Generate QC image"}
    step['command']="micromamba run -n base python {g_pymipl_dir}/slice_qc.py  \
        -o {job_workdir}/qc.png --mask {job_workdir}/lesion_mask.nii.gz {job_workdir}/ct_struct.nii"
    job['steps']+=[step]

    step={"step_title": "4. Clean up"}
    step['step_command']="rm -r {job_workdir}/ct_struct.nii {job_workdir}/DICOM"
    job['steps']+=[step]

    step={"step_title": "5. Upload results to XNAT"}
    step['step_command']='micromamba run -n base python {g_pymipl_dir}/xnat_workflow/sync-resource-with-xnat.py \
            --level experiment --project {g_project} --subject {job_subject} --experiment {job_exp_label} --local_resource {job_workdir} \
            --remote_resource {g_workflow_id} --create_hierarchy 1'
    job['steps']+=[step]

    ####################################################################################################
    # This block processes job and writes configuration and the final script.
    job_id=f"{workflow_id}_{job['Subject']}_{job['Experiment']}"
    local_job_dir=local_workdir_path / 'jobs' / job_id
    job_file_yaml=local_job_dir / 'job.yaml'
    job_file_sh=local_job_dir / 'job.sh'
    local_job_dir.mkdir(parents=True,exist_ok=True)
    
    with open(job_file_yaml,"w") as f:
        yaml.safe_dump(paths_to_str(job),f,sort_keys=False)

    #this also needs to be a dedicated function. 
    if env_type=='jupyter': #write all commands to a single batch file
        wa.workflow_to_batch(job,global_vars,batch_file)
        print(batch_file)
        break #DEBUG
        
    else: #one batch file per job
        #create batch        
        print(job_file_sh)
        #reset job script
        ! truncate -s 0 {job_file_sh}
        #generate job script
        wa.workflow_to_batch(job,global_vars,job_file_sh)
        ! chmod +x {job_file_sh}

        ########################################################################################
        # Finally, this stores the batch file to XNAT resource and launches dedicated container.
        print('sending batch to xnat resource')
        if xnat_interface is None: xnat_interface=get_xnat_interface(project)
        #DEBUG: uncomment the next line later.
        res1=wa.resource_to_xnat(local_job_dir, workflow_id, project, job['job_subject'], job['job_exp_label'],xnat_interface)
        if res1 != 0: raise ConnectionError("Uploading job configuration to XNAT failed")            
        print('submitting job to Container Service')
        # Retreive experiment ID from the experiment label. This is required to launch container.
        exp_id=xnat_interface.select.project(project).subject(job['Subject']).experiment(job['job_exp_label']).id()
        res2=1 #DEBUG - uncomment the next line to actually launch.
        #res2=wa.launch_cs_command(xnat_interface,project,job['Subject'],job['Experiment'],exp_id,workflow_id,xnat_command_id,xnat_command_wrapper_id,verbose=True)                
        if not res2: raise ConnectionError("Launching container job failed")
        if n % 10 == 0: print(f'Done {n} out of {num_sessions} ({n*100/num_sessions:.1f}%)')


NameError: name 'sys' is not defined

In [9]:
!pip install nibabel==5.3.0
#interface=get_xnat_interface(project)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 55.9 MB/s eta 0:00:00a 0:00:01


In [13]:
exp=xnat_interface.select.project(project).subject('LUNG1-093').experiment('04-13-2006-StudyID-NA-25111').id()

2026-03-26 20:01:32,219 - urllib3.connectionpool - DEBUG - https://tap.embarklabs.ai:443 "GET /data/projects/NSCLC_RADIOMICS/subjects/LUNG1-093/experiments?format=csv&columns=ID,label HTTP/1.1" 200 None
2026-03-26 20:01:32,219 - urllib3.connectionpool - DEBUG - https://tap.embarklabs.ai:443 "GET /data/projects/NSCLC_RADIOMICS/subjects/LUNG1-093/experiments?format=csv&columns=ID,label HTTP/1.1" 200 None


In [ ]:
pip install --force-reinstall --no-cache-dir "numpy<2" pandas matplotlib

In [29]:
!pip cache purge

Files removed: 6
